In [1]:
!pip install xgboost lightgbm catboost

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                              ExtraTreesClassifier, HistGradientBoostingClassifier, 
                              BaggingClassifier, AdaBoostClassifier)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [3]:
df_train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
df_test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

In [4]:
df_train = df_train.set_index(['PassengerId'])
df_test = df_test.set_index(['PassengerId'])

In [5]:
df_train.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
PassengerId,,,,,,,,,,,,,
0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8693 entries, 0001_01 to 9280_02
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8492 non-null   object 
 1   CryoSleep     8476 non-null   object 
 2   Cabin         8494 non-null   object 
 3   Destination   8511 non-null   object 
 4   Age           8514 non-null   float64
 5   VIP           8490 non-null   object 
 6   RoomService   8512 non-null   float64
 7   FoodCourt     8510 non-null   float64
 8   ShoppingMall  8485 non-null   float64
 9   Spa           8510 non-null   float64
 10  VRDeck        8505 non-null   float64
 11  Name          8493 non-null   object 
 12  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(6)
memory usage: 891.4+ KB


In [7]:
df_train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [8]:
def add_engineered_features(df):
    df = df.copy()
    
    df['AllSpentMoney'] = df['RoomService'] + df['FoodCourt'] + df['ShoppingMall'] + df['Spa'] + df['VRDeck']

    df['AllSpentMoneyToAge'] = df['AllSpentMoney'] / df['Age']
    
    return df

df_train = add_engineered_features(df_train)
df_test = add_engineered_features(df_test)

In [9]:
num_features = df_train.select_dtypes(include=['float64']).columns.tolist()
cat_features = df_train.select_dtypes(include=['object']).columns.tolist()

all_features = num_features + cat_features

print(f"cat: {cat_features}")
print(f"\nnum: {num_features}")

cat: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'Name']

num: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'AllSpentMoney', 'AllSpentMoneyToAge']


In [10]:
numeric_data = df_train[num_features]
corr_matrix = numeric_data.corr().abs()

upper_triangle = corr_matrix.where(np.triu(np.ones(shape=corr_matrix.shape), k=1).astype(bool))

features_to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]

print(f"Найдено {len(features_to_drop)} сильно скоррелированных признаков для удаления: {features_to_drop}")

for col in features_to_drop:
    if col in num_features:
        num_features.remove(col)
    if col in all_features:
        all_features.remove(col)

print(f"Осталось признаков для обучения: {len(all_features)}")

Найдено 1 сильно скоррелированных признаков для удаления: ['AllSpentMoneyToAge']
Осталось признаков для обучения: 13


In [11]:
correlations = numeric_data.corrwith(df_train['Transported'])
correlations

Age                  -0.075026
RoomService          -0.244611
FoodCourt             0.046566
ShoppingMall          0.010141
Spa                  -0.221131
VRDeck               -0.207075
AllSpentMoney        -0.197671
AllSpentMoneyToAge   -0.192714
dtype: float64

In [12]:
X = df_train[all_features]
y = df_train['Transported']

In [13]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

In [14]:
#models_to_compare = {
#    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
#    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
#    "HistGradientBoostingClassifier": HistGradientBoostingClassifier(random_state=42),
#    "BaggingClassifier": BaggingClassifier(random_state=42),
#    "AdaBoostClassifier": AdaBoostClassifier(random_state=42),
#
#    "XGBoost": XGBClassifier(n_estimators=100, random_state=42),
#    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42),
#    "CatBoost": CatBoostClassifier(iterations=100, random_state=42, verbose=0)
#}
#
#results = []
#
#for name, model in models_to_compare.items():
#    try:
#        pipeline = Pipeline(steps=[
#            ('preprocessor', preprocessor),
#            ('model', model)
#        ])
#    
#        scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy', n_jobs=-1)
#            
#        results.append({
#            'model': name,
#            'accuracy_score': scores.mean(),
#            'std_score': scores.std()
#        })
#    except Exception as e:
#        print(f"model: {model}, error: {e}")
#
#results_df = pd.DataFrame(results).sort_values(by='accuracy_score', ascending=False)

In [15]:
#results_df

In [16]:
import optuna
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import time

def objective(trial):
    start = time.time()
    
    params = {
        'max_iter': trial.suggest_int('max_iter', 100, 300),  
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 50),
        'l2_regularization': trial.suggest_float('l2_regularization', 0, 1),
        'max_bins': trial.suggest_int('max_bins', 128, 255),
    }
    
    model = HistGradientBoostingClassifier(
        **params, 
        random_state=42,
        early_stopping=True,  
        validation_fraction=0.1
    )
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    scores = cross_val_score(
        pipeline, X, y, 
        cv=5,           
        scoring='accuracy', 
        n_jobs=1
    )
    
    print(f"Trial {trial.number}: accuracy={scores.mean():.4f}, time={time.time()-start:.1f}s")
    
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n✅ Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"✅ Лучшая accuracy: {study.best_value:.4f}")

[I 2026-06-30 08:40:53,866] A new study created in memory with name: no-name-74156430-3d15-40d6-91a0-c4d24385772f


  0%|          | 0/30 [00:00<?, ?it/s]

Trial 0: accuracy=0.7931, time=614.1s
[I 2026-06-30 08:51:07,976] Trial 0 finished with value: 0.7930545510130751 and parameters: {'max_iter': 296, 'learning_rate': 0.016897663187722834, 'max_depth': 4, 'min_samples_leaf': 25, 'l2_regularization': 0.8422468023952399, 'max_bins': 212}. Best is trial 0 with value: 0.7930545510130751.
Trial 1: accuracy=0.7945, time=182.7s
[I 2026-06-30 08:54:10,695] Trial 1 finished with value: 0.7945495308005408 and parameters: {'max_iter': 162, 'learning_rate': 0.10813005081605609, 'max_depth': 3, 'min_samples_leaf': 24, 'l2_regularization': 0.002324602660148778, 'max_bins': 161}. Best is trial 1 with value: 0.7945495308005408.
Trial 2: accuracy=0.7962, time=529.8s
[I 2026-06-30 09:03:00,474] Trial 2 finished with value: 0.7961601147704029 and parameters: {'max_iter': 204, 'learning_rate': 0.025162157682640993, 'max_depth': 5, 'min_samples_leaf': 43, 'l2_regularization': 0.03339980875358439, 'max_bins': 217}. Best is trial 2 with value: 0.79616011477040

In [17]:
print(study.best_params)
print(f"Best accuracy: {study.best_value:.4f}")

optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)

{'max_iter': 254, 'learning_rate': 0.12229777533749973, 'max_depth': 8, 'min_samples_leaf': 34, 'l2_regularization': 0.05437678870983234, 'max_bins': 206}
Best accuracy: 0.7974


In [18]:
best_model = HistGradientBoostingClassifier(**study.best_params, random_state=42)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_model)
])

final_pipeline.fit(X, y)

test_predictions = final_pipeline.predict(df_test[all_features])

In [19]:
submission = pd.DataFrame({
    'PassengerId': df_test.index,  
    'Transported': test_predictions
})

submission.to_csv('submission.csv', index=False)